Ao rodar o experimento do SFT - Cot, houve um erro no salvamento do modelo. Esse notebook serve para resolver os problemas gerados, a partir dos checkpoints salvos pelo unsloth

Recuperação e Salvamento do Modelo

In [ ]:
from unsloth import FastLanguageModel
from pathlib import Path

# Aponta para o último checkpoint salvo pelo SFTTrainer
checkpoint_dir = Path("../models/sft_cot_3class_qwen/checkpoint-600")  # substitui pelo número real

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(checkpoint_dir),
    max_seq_length=2560,   # mesmo valor do treino
    dtype=None,
    load_in_4bit=True,
)

output_lora = Path(checkpoint_dir).parent / "lora_adapters"
output_merged = Path(checkpoint_dir).parent / "merged_model"

# Tenta merged_4bit_forced primeiro; cai para lora_merged_16bit se falhar
for method in ["merged_4bit_forced", "lora_merged_16bit", "merged_16bit"]:
    model.save_pretrained(str(output_lora))
    tokenizer.save_pretrained(str(output_lora))
    try:
        print(f"Tentando save_method='{method}'...")
        model.save_pretrained_merged(str(output_merged), tokenizer, save_method=method)
        print(f"Sucesso com '{method}'. Modelo salvo em: {output_merged}")
        tokenizer.save_pretrained(str(output_merged))
        break
    except Exception as e:
        print(f"'{method}' falhou: {e}")

Recuperação e Salvamento das Métricas

In [ ]:
# recover_metrics.py
import json
import pandas as pd
from pathlib import Path
from datetime import timedelta, datetime

# --- Configs manuais ---
exp_name    = "qwen_sft_cot_3class_(recovered)"   # ajuste com seu model_prefix + mode
adapter_name = "sft_cot_3class_qwen"               # mesmo valor que get_adapter_name retornaria
n_samples   = 17268                    # tamanho do train_df, se lembrar

# --- trainer_state.json ---
state_path = Path("/home/augusto/paper_2026/experiments/paper2026/models/sft_cot_3class_qwen/checkpoint-717/trainer_state.json")
with open(state_path) as f:
    state = json.load(f)

log_history  = state["log_history"]
train_losses = [
    {"step": e["step"], "loss": e["loss"]}
    for e in log_history if "loss" in e and "eval_loss" not in e
]
eval_losses = [
    {"step": e["step"], "eval_loss": e["eval_loss"]}
    for e in log_history if "eval_loss" in e
]

# --- CodeCarbon CSV ---
row = pd.read_csv("/home/augusto/paper_2026/experiments/paper2026/emissions.csv").iloc[-1]  # última linha = sft_cot

elapsed   = row["duration"]
end_time   = datetime.fromisoformat(row["timestamp"])
start_time = end_time - timedelta(seconds=elapsed)
throughput = round(n_samples / elapsed, 4) if elapsed and n_samples else 0.0

resource_stats = {
    "experiment":                exp_name,
    "start_time":                start_time.timestamp(),   
    "end_time":                  end_time.timestamp(),
    "elapsed_seconds":           round(elapsed, 3),
    "n_samples":                 n_samples,
    "throughput_samples_per_sec": throughput,
    "peak_vram_mb":              83655.3,   # alterar de acordo com o log
    "peak_ram_mb":               106148.9,   # alterar de acordo com o log
    "emissions_kg_co2":          row["emissions"],
}

metrics = {
    "experiment":            exp_name,
    "adapter":               adapter_name,
    "train_loss_history":    train_losses,
    "eval_loss_history":     eval_losses,
    "train_runtime_seconds": elapsed,
    "resource_stats":        resource_stats,
}

output = Path("/home/augusto/paper_2026/experiments/paper2026/outputs/results/qwen_sft_cot_3class_training_metrics.json")
output.parent.mkdir(parents=True, exist_ok=True)
with open(output, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print("Salvo em:", output)

GRPO -> chave em vez de ser mean_reward era apenas reward. Recuperação do histórico via checkpoint

In [ ]:
# recover_grpo_metrics.py
import json
from pathlib import Path

state_path = Path("../models/grpo_standard_3class_qwen/checkpoint-2158/trainer_state.json")
metrics_path = Path("../outputs/results/qwen_grpo_standard_3class_training_metrics.json")

# Carrega o trainer_state
with open(state_path) as f:
    state = json.load(f)

# Extrai o histórico de reward
mean_reward_history = []
for entry in state["log_history"]:
    if "reward" in entry:
        mean_reward_history.append({
            "step": entry.get("step", 0),
            "mean_reward": entry["reward"],
        })

# Carrega o metrics existente e atualiza só o mean_reward_history
with open(metrics_path) as f:
    metrics = json.load(f)

metrics["mean_reward_history"] = mean_reward_history

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f"Salvo em: {metrics_path}")
print(f"Steps recuperados: {len(mean_reward_history)}")
print(f"Reward final: {mean_reward_history[-1]['mean_reward']:.4f}")